# 04 - Optimización de Hiperparámetros

## Optimización de Hiperparámetros

**Objetivo:** Mejorar los modelos base mediante búsqueda sistemática de hiperparámetros usando `GridSearchCV`.

**Modelos optimizados:** Random Forest y Gradient Boosting  
**Método:** GridSearchCV con StratifiedKFold 5-fold y scoring `f1_weighted`

In [ ]:
import pandas as pd, numpy as np, matplotlib.pyplot as plt, seaborn as sns
from sklearn.preprocessing import StandardScaler
from sklearn.cluster import KMeans
from sklearn.metrics import silhouette_score
from sklearn.model_selection import train_test_split, GridSearchCV, StratifiedKFold
from sklearn.ensemble import RandomForestClassifier, GradientBoostingClassifier
from sklearn.metrics import accuracy_score
import pickle, warnings
warnings.filterwarnings('ignore')
sns.set_style('whitegrid')
np.random.seed(42)
print(' Setup')

✓ Setup


In [ ]:
# Reproducir 02
df = pd.read_csv('../datos/llm_price_performance_tracker_2026-03-31.csv')
df_limpio = df.dropna(subset=['aa_intelligence_index', 'aa_coding_index'], how='all').copy()
df_limpio.loc[df_limpio['is_open_source'] == True, 'input_cost_usd_per_1m'] = df_limpio.loc[df_limpio['is_open_source'] == True, 'input_cost_usd_per_1m'].fillna(0)
df_limpio.loc[df_limpio['is_open_source'] == True, 'output_cost_usd_per_1m'] = df_limpio.loc[df_limpio['is_open_source'] == True, 'output_cost_usd_per_1m'].fillna(0)

columnas = ['aa_intelligence_index', 'aa_coding_index', 'aa_math_index', 'input_cost_usd_per_1m', 'output_cost_usd_per_1m', 'output_tokens_per_second', 'time_to_first_token_s', 'chatbot_arena_elo', 'release_year']
for col in columnas:
    if col in df_limpio.columns:
        df_limpio[col] = df_limpio.groupby('provider')[col].transform(lambda x: x.fillna(x.median()))
        df_limpio[col] = df_limpio[col].fillna(df_limpio[col].median())

df_limpio['costo_promedio'] = (df_limpio['input_cost_usd_per_1m'] + df_limpio['output_cost_usd_per_1m']) / 2
df_limpio['inteligencia_por_dolar'] = df_limpio['aa_intelligence_index'] / (df_limpio['costo_promedio'] + 0.001)
df_limpio['velocidad_por_dolar'] = df_limpio['output_tokens_per_second'] / (df_limpio['costo_promedio'] + 0.001)
df_limpio['promedio_benchmarks'] = (df_limpio['aa_intelligence_index'] + df_limpio['aa_coding_index'] + df_limpio['aa_math_index']) / 3
df_limpio['ratio_valor_general'] = df_limpio['promedio_benchmarks'] / (df_limpio['costo_promedio'] + 0.001)

features = ['aa_intelligence_index', 'aa_coding_index', 'aa_math_index', 'costo_promedio', 'output_tokens_per_second', 'inteligencia_por_dolar', 'velocidad_por_dolar', 'ratio_valor_general', 'chatbot_arena_elo', 'release_year']
X = df_limpio[features].copy()
scaler = StandardScaler()
X_escalado = scaler.fit_transform(X)

silhouette_scores = []
for k in range(2, 10):
    kmeans = KMeans(n_clusters=k, random_state=42, n_init=10)
    kmeans.fit(X_escalado)
    sil_score = silhouette_score(X_escalado, kmeans.labels_)
    silhouette_scores.append(sil_score)

k_optimal = 2 + np.argmax(silhouette_scores)
kmeans_final = KMeans(n_clusters=k_optimal, random_state=42, n_init=10)
df_limpio['cluster'] = kmeans_final.fit_predict(X_escalado)

df_limpio['nuestro_pricing_tier'] = df_limpio['pricing_tier'].copy()
for idx, row in df_limpio[df_limpio['pricing_tier'] == 'Unknown'].iterrows():
    cluster_id = row['cluster']
    cluster_data = df_limpio[(df_limpio['cluster'] == cluster_id) & (df_limpio['pricing_tier'] != 'Unknown')]
    if len(cluster_data) > 0:
        most_common = cluster_data['pricing_tier'].mode()
        if len(most_common) > 0:
            df_limpio.loc[idx, 'nuestro_pricing_tier'] = most_common[0]

X_train, X_test, y_train, y_test = train_test_split(X, df_limpio['nuestro_pricing_tier'], test_size=0.2, random_state=42, stratify=df_limpio['nuestro_pricing_tier'])
print(' Datos listos')

✓ Datos listos


## GridSearchCV — Random Forest

**Espacio de búsqueda:**

| Parámetro | Valores probados |
|-----------|-----------------|
| `n_estimators` | 50, 100, 150 |
| `max_depth` | 5, 10, 15, 20 |
| `min_samples_split` | 2, 5, 10 |
| `min_samples_leaf` | 1, 2, 4 |

**Total de combinaciones:** 3 × 4 × 3 × 3 = **108 configuraciones**  
**Evaluaciones totales (con CV 5-fold):** 540

In [ ]:
print('GridSearchCV RF...')
param_grid_rf = {'n_estimators': [50, 100, 150], 'max_depth': [5, 10, 15, 20], 'min_samples_split': [2, 5, 10], 'min_samples_leaf': [1, 2, 4]}
grid_search_rf = GridSearchCV(RandomForestClassifier(random_state=42, n_jobs=-1), param_grid_rf, cv=StratifiedKFold(n_splits=5), scoring='f1_weighted', n_jobs=-1)
grid_search_rf.fit(X_train, y_train)
rf_opt = grid_search_rf.best_estimator_
print(f' RF CV score: {grid_search_rf.best_score_:.4f}')
print(f'Test: {accuracy_score(y_test, rf_opt.predict(X_test)):.4f}')

GridSearchCV RF...
✓ RF CV score: 0.9667
Test: 0.9889


**Resultado:**

| Métrica | Valor |
|---------|-------|
| Mejor CV F1-weighted | **0.9667** |
| Test Accuracy (optimizado) | **0.9889** |

> El proceso de optimización encontró que el modelo base ya estaba razonablemente configurado. La mejora en CV es modesta (+0.5% F1), lo que confirma que los patrones del dataset son robustos e independientes de configuraciones específicas

## GridSearchCV — Gradient Boosting

**Espacio de búsqueda:**

| Parámetro | Valores probados |
|-----------|-----------------|
| `n_estimators` | 50, 100, 150 |
| `max_depth` | 3, 5, 7 |
| `learning_rate` | 0.01, 0.05, 0.1 |
| `min_samples_split` | 2, 5, 10 |

**Total de combinaciones:** 3 × 3 × 3 × 3 = **81 configuraciones**  
**Evaluaciones totales:** 405

In [ ]:
print('GridSearchCV GB...')
param_grid_gb = {'n_estimators': [50, 100, 150], 'max_depth': [3, 5, 7], 'learning_rate': [0.01, 0.05, 0.1], 'min_samples_split': [2, 5, 10]}
grid_search_gb = GridSearchCV(GradientBoostingClassifier(random_state=42), param_grid_gb, cv=StratifiedKFold(n_splits=5), scoring='f1_weighted', n_jobs=-1)
grid_search_gb.fit(X_train, y_train)
gb_opt = grid_search_gb.best_estimator_
print(f' GB CV score: {grid_search_gb.best_score_:.4f}')
print(f'Test: {accuracy_score(y_test, gb_opt.predict(X_test)):.4f}')

GridSearchCV GB...
✓ GB CV score: 0.9772
Test: 0.9889


**Resultado:**

| Métrica | Valor |
|---------|-------|
| Mejor CV F1-weighted | **0.9772** |
| Test Accuracy (optimizado) | **0.9889** |

> GB optimizado obtiene el **mayor CV F1-weighted del proyecto (0.9772)**. La mejora respecto al modelo base (+1.08% en F1 CV) indica que la optimización de hiperparámetros tuvo un efecto más significativo en GB que en RF.

## Serialización de Modelos Optimizados

Los modelos entrenados se guardan usando `pickle` para:
- Reutilización sin reentrenamiento
- Reproducibilidad de resultados
- Aplicación en el análisis final (Notebook 05)

**Archivos generados:**

| Archivo | Contenido |
|---------|-----------|
| `rf_optimizado.pkl` | Random Forest con mejores hiperparámetros |
| `gb_optimizado.pkl` | Gradient Boosting con mejores hiperparámetros |
| `scaler.pkl` | StandardScaler ajustado a datos de entrenamiento |

In [ ]:
# Guardar
with open('../models/trained_models/rf_optimizado.pkl', 'wb') as f: pickle.dump(rf_opt, f)
with open('../models/trained_models/gb_optimizado.pkl', 'wb') as f: pickle.dump(gb_opt, f)
with open('../models/trained_models/scaler.pkl', 'wb') as f: pickle.dump(scaler, f)
print(' Modelos guardados')

✓ Modelos guardados


## Conclusiones — Optimización de Hiperparámetros

**Comparativa base vs optimizado:**

| Modelo | CV F1 base | CV F1 optimizado | Mejora |
|--------|-----------|-----------------|--------|
| Random Forest | 0.9609 (acc) | 0.9667 (F1w) | +0.58% |
| Gradient Boosting | 0.9664 (acc) | **0.9772 (F1w)** | +1.08% |

**Conclusión:** La optimización confirma que **Gradient Boosting** es el mejor modelo de clasificación del proyecto, con F1-weighted de 0.9772 en validación cruzada. Ambos modelos mantienen 98.89% de accuracy en test, lo que indica que los datos tienen una estructura altamente predecible dado el pipeline de clustering aplicado.